In [0]:
from pyspark.sql import functions as F

In [0]:
seasons = ['2122', '2223', '2324', '2425', '2526']
leagues = ['bundesliga', 'la_liga', 'ligue_one', 'premier_league', 'serie_a']
for season in seasons:
    for league in leagues:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(f"/Volumes/football_data_platform/raw/source_files/football_data/{season}/{season}_{league}.csv")
        )

        bronze_df = (
            df
            .withColumn("_season", F.lit(season))
            .withColumn("_league", F.lit(league))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_source_system", F.lit("football-data.co.uk"))
        )

        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"football_data_platform.bronze.football_data_{league}")
        )